# Country vulnerability analysis

This notebook aggregates pixel-level vulnerability results to the country level. It uses the raster of overall vulnerability together with global country boundaries to compute national statistics. In addition, it loads the normalized variables used to construct the sensitivity and lack of adaptive capacity dimensions in order to calculate country-level contributions to vulnerability of each variable.

The workflow first aligns the country boundaries with the vulnerability raster and creates a country identifier raster matching the raster grid. Using this identifier raster, the notebook then extracts pixel values belonging to each country and calculates summary statistics. It then creates:

- a raster with country identifiers aligned to the vulnerability raster grid
- a lookup table linking raster country IDs to country names
- a table with country-level statistics of overall vulnerability
- a table with country-level statistics for each normalized variable contributing to vulnerability score
- A stacked pillar diagram showing 10 most vulnerable countries with variable contributions
- A map showing mean vulnerability classes for countries
  
## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `vulnerability_mean.tif`
- `lackof_adapt_mean.tif`
- `bii_5000m.tif`
- `wdpa_5000m.tif`
- `landmark_5000m.tif`
- `kba_5000m.tif`
- `poverty_5000m.tif`
- `water_risk_5000m.tif`
- `conflict_5000m.tif`
- `edi_5000m.tif` 
- `landrights_5000m.tif`
- `rule_of_law_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)


In [ ]:
# Configuration (edit these paths if needed)

#input paths
VULN = 'vulnerability_indicator\\vulnerability_mean.tif'
WORLD_COUNTRIES_GENERAL = 'lackof_adapt\\World_Countries_(Generalized)_8414823838130214587.gpkg' 


sens_vars = {
    "Biodiversity Intactness":    'sensitivity\\biodiversity_intactness\\bii_5000m.tif',
    "Protected Areas":  'sensitivity\\protected_areas\\wdpa_5000m.tif',
    "IPLC Lands": 'sensitivity\\ind_com_lands\\landmark_5000m.tif' ,
    "Key Biodiversity Areas":'sensitivity\\kba\\kba_5000m.tif',
    "Poverty":   'sensitivity\\poverty\\poverty_5000m.tif',
    "Water Risk": 'sensitivity\\water_risk\\water_risk_5000m.tif',
}

lackof_adapt_vars= {
    "Conflict":  'lackof_adapt\\conflict\\conflict_5000m.tif',
    "Environmental Democracy": 'lackof_adapt\\environmental_democracy\\edi_5000m.tif',
    "Landrights":'lackof_adapt\\landrights\\landrights_5000m.tif',
    "Rule of Law":   'lackof_adapt\\rule_of_law\\rule_of_law_5000m.tif'
}

# output paths
COUNTRY_ID = 'vulnerability_indicator\\countries\\country_id.tif'  
LOOK_UP = 'vulnerability_indicator\\countries\\country_id_lookup.csv' 
STATS = 'vulnerability_indicator\\countries\\country_vulnerability_stats.csv'
CONTRIBUTION= "country_variable_contrib.csv"

#other parameters
#define window size and nodata
window_size = 2048
dst_nodata = -9999.0

# Set threshold for high vulnerability
threshold_mode = "global_quantile"  # global_quantile
global_quantile = 0.75              

# percentile to report
pctl = 0.90

# histogram settings for approximate percentiles
n_bins = 500

#column names of country boundary data
iso_col = "ISO"
name_col = "COUNTRY"


In [ ]:
#import packages
import pandas as pd
import geopandas as gpd
import math
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio import warp
from rasterio import features
import geopandas as gpd
from shapely.geometry import box
from rasterio.features import geometry_mask
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from rasterio.features import rasterize
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

In [ ]:
#calculate country stats (4 steps)
# 1)COUNTRY IDENTIFIER FUNCTION

def build_country_id_raster(ref_path: str, gpkg_path: str, out_raster: str, out_lookup_csv: str):
    #open vulnerability raster as reference raster
    with rasterio.open(ref_path) as ref:
        ref_meta = ref.meta.copy()
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
    #open gpkg with country boundaries
    gdf = gpd.read_file(gpkg_path).to_crs(ref_crs)

    # drop Antarctica
    if name_col and "Antarctica" in set(gdf[name_col].dropna().unique()):
        gdf = gdf[gdf[name_col] != "Antarctica"].copy()

    # create  country keys 
    gdf["_key"] = gdf[iso_col].astype("string")
    #sort 
    keys = sorted(gdf["_key"].unique().tolist())
    #apply id numbers in dictionary
    key_to_id = {k: i + 1 for i, k in enumerate(keys)}
    gdf["_id"] = gdf["_key"].map(key_to_id).astype(np.int32)#look up values in dictionary

   
    tmp = gdf.drop_duplicates("_key")[["_key", name_col]].copy()
    
    lookup = pd.DataFrame({
        "country_id": [key_to_id[k] for k in keys],
        "iso": keys,
    })

    #add name column
    tmp=tmp.rename(columns={"_key": "iso", name_col: "name"})

    #merge to also add name column
    lookup=lookup.merge(
        tmp,
        on="iso",
        how="left",
    )
    #save as csv
    lookup.to_csv(out_lookup_csv, index=False)

    # spatial index to select candidates of polygons that might intersect window
    sindex = gdf.sindex

    # define raster meta using the reference raster and update
    out_meta = ref_meta.copy()
    out_meta.update(
        dtype="int32",
        count=1,
        nodata=0,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256,
    )
    #open output raster
    with rasterio.open(out_raster, "w", **out_meta) as dst:
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                win_bounds = rasterio.windows.bounds(window, ref_transform)
                win_geom = box(*win_bounds)

                # slight buffer to avoid missing edges of countries
                px = max(abs(ref_transform.a), abs(ref_transform.e))
                win_geom = win_geom.buffer(px)
                #select candidates
                cand_idx = list(sindex.intersection(win_geom.bounds))
                if not cand_idx: #if there are no candidates, continue
                    dst.write(np.zeros((h, w), dtype=np.int32), 1, window=window)
                    continue
                #select polygon subset of candidates that actually intersect
                sub = gdf.iloc[cand_idx]
                sub = sub[sub.intersects(win_geom)]
                if sub.empty:
                    dst.write(np.zeros((h, w), dtype=np.int32), 1, window=window)
                    continue

                win_transform = rasterio.windows.transform(window, ref_transform)
                shapes = ((geom, int(cid)) for geom, cid in zip(sub.geometry, sub["_id"]))#write correct country id to geometry, cid=country id
                #rasterize
                burned = features.rasterize(
                    shapes=shapes,
                    out_shape=(h, w),
                    transform=win_transform,
                    fill=0,
                    all_touched=False,
                    dtype="int32",
                )
                #write raster
                dst.write(burned, 1, window=window)

    print(f"{out_raster}")
    print(f"{out_lookup_csv}")

In [ ]:
# 2) FUNCTION TO IDENTIFY GLOBAL MAX/MIN (important for relative vulnerability/global comparisons)
def compute_global_minmax(VULN: str, country_id_path: str):
    #open rasters of vulnersbility and country_id
    with rasterio.open(VULN) as vsrc, rasterio.open(country_id_path) as csrc:
        ref_width, ref_height = vsrc.width, vsrc.height

        vmin = np.inf #largest possible minimum
        vmax = -np.inf #smallest possible maximum

        #calculate number of windows
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                v = vsrc.read(1, window=window)
                cid = csrc.read(1, window=window)

                #define valid cells
                valid = (cid > 0) & np.isfinite(v) & (v != dst_nodata)#valid if inside countries, not no data, and finite number

                if np.any(valid):
                    vv = v[valid].astype(np.float32) #filter for only valid values
                    vmin = min(vmin, float(vv.min())) #update min out of valid values
                    vmax = max(vmax, float(vv.max()))#update max out of valid values
    #return max and min              
    return vmin, vmax

In [ ]:
# 3) FUNCTION TO DETERMINE THRESHOLD FOR HIGH VULNERABILITY

def global_threshold_from_hist(VULN: str, country_id_path: str, vmin: float, vmax: float, q: float):
    edges = np.linspace(vmin, vmax, n_bins + 1, dtype=np.float32)#define edges
    hist = np.zeros(n_bins, dtype=np.int64)
    #open rasters
    with rasterio.open(VULN) as vsrc, rasterio.open(country_id_path) as csrc:
        ref_width, ref_height = vsrc.width, vsrc.height
        #calculate number of windows
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                v = vsrc.read(1, window=window)
                cid = csrc.read(1, window=window)
                
                #define valid cells
                valid = (cid > 0) & np.isfinite(v) & (v != dst_nodata)#valid if inside countries, not no data, and finite number
                if not np.any(valid):
                    continue

                vv = v[valid].astype(np.float32)#filter for only valid values
                histwin, _ = np.histogram(vv, bins=edges)
                hist += histwin.astype(np.int64)

    total = hist.sum()
    
    target = int(math.ceil(q * total))
    cumulative_sum = 0
    for i, count in enumerate(hist):
        cumulative_sum += int(count)
        if cumulative_sum >= target:
            # return left edge of the bin as a conservative threshold, stops when cumulative sum reaches the target percentile
            return float(edges[i])

# 3.1) FUNCTION TO CONVERT HISTOGRAM IN PERCENTILE VALUES
def percentile_from_country_hist(hist: np.ndarray, edges: np.ndarray, q: float):
    # total number of valid pixels for this country
    total = int(hist.sum())
    # if there are no pixels, percentile cannot be computed
    if total == 0:
        return np.nan
    #convert percentil in target pixel position
    target = int(math.ceil(q * total))
    cumulative_sum = 0
    #loop over histogram bins from lowest values to highest
    for i, count in enumerate(hist):
        # add number of pixels in this bin
        cumulative_sum += int(count)
        if cumulative_sum >= target:
            # bin boundaries (value range of this bin)
            left = float(edges[i])
            right = float(edges[i + 1])
            #number of pixels counted before this bin
            prev_cumulative_sum = cumulative_sum - int(count)
            #check
            if count == 0:
                return left
            #fraction of the bin needed to reach the target pixel
            frac = (target - prev_cumulative_sum) / count
            #interpolate inside the bin to estimate percentile value
            return left + frac * (right - left)

In [ ]:
# 4)FUNCTION TO COMPUTE COUNTRY STATISTICS

def compute_country_stats(
    VULN: str,
    country_id_path: str,
    lookup_csv: str,
    out_csv: str,
    vmin: float,
    vmax: float,
    pctl: float,
    high_threshold: float,
):
    """
    Compute per country:
      - mean vulnerability (pixel mean == area-weighted mean for equal-area raster)
      - variance, std dev, coefficient of variation (CV)
      - 90th percentile (approx from histogram)
      - % area above threshold
      -skeweness
      -excess kurtosis
    """
    
    edges = np.linspace(vmin, vmax, n_bins + 1, dtype=np.float32)

    #load lookup table
    lookup = pd.read_csv(lookup_csv)
    #create list from country_id
    country_ids = lookup["country_id"].astype(int).tolist()
    #find count of countries by reading max country id
    max_id = max(country_ids) 

    # Prepare accumulators
    sum_by_id = np.zeros(max_id + 1, dtype=np.float64)
    sumsq_by_id = np.zeros(max_id + 1, dtype=np.float64)  
    count_by_id = np.zeros(max_id + 1, dtype=np.int64)
    above_by_id = np.zeros(max_id + 1, dtype=np.int64)
    hist_by_id = np.zeros((max_id + 1, n_bins), dtype=np.int64)
    sumcube_by_id = np.zeros(max_id + 1, dtype=np.float64)
    sumquad_by_id = np.zeros(max_id + 1, dtype=np.float64)
    #open rasters
    with rasterio.open(VULN) as vsrc, rasterio.open(country_id_path) as csrc:
        ref_width, ref_height = vsrc.width, vsrc.height
        #calculate number of windows
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                v = vsrc.read(1, window=window)
                cid = csrc.read(1, window=window)
                #define valid values
                valid = (cid > 0) & np.isfinite(v) & (v != dst_nodata)#valid if inside countries, not no data, and finite number
                if not np.any(valid):
                    continue

                vv = v[valid].astype(np.float32) #filter for valid values
                cc = cid[valid].astype(np.int32)# matching country ids
                # sum(v^3)
                sumcube_by_id += np.bincount(
                    cc,
                    weights=(vv.astype(np.float64) ** 3),
                    minlength=max_id + 1
                )
                # sum(v^4)
                sumquad_by_id += np.bincount(
                    cc,
                    weights=(vv.astype(np.float64) ** 4),
                    minlength=max_id + 1
                )


                # sum(v)
                sum_by_id += np.bincount(cc, weights=vv.astype(np.float64), minlength=max_id + 1)

                # sum(v^2)  (NEW)
                sumsq_by_id += np.bincount(cc, weights=(vv.astype(np.float64) ** 2), minlength=max_id + 1)

                def _skewness(i):
                    i = int(i)
                    n = count_by_id[i]
                    if n <= 2:
                        return np.nan
                
                    mean = sum_by_id[i] / n
                    m2 = (sumsq_by_id[i] / n) - mean**2
                    if m2 <= 0:
                        return np.nan
                
                    m3 = (sumcube_by_id[i] / n) - 3*mean*m2 - mean**3
                    skew = m3 / (m2 ** 1.5)
                    return float(skew)

                def _kurtosis(i):
                        i = int(i)
                        n = count_by_id[i]
                        if n <= 3:
                            return np.nan
                    
                        mean = sum_by_id[i] / n
                        m2 = (sumsq_by_id[i] / n) - mean**2
                        if m2 <= 0:
                            return np.nan
                    
                        m4 = (
                            (sumquad_by_id[i] / n)
                            - 4*mean*(sumcube_by_id[i] / n)
                            + 6*(mean**2)*(sumsq_by_id[i] / n)
                            - 3*(mean**4)
                        )
                    
                        kurt = m4 / (m2 ** 2)
                        excess_kurt = kurt - 3.0   # excess kurtosis (recommended)
                        return float(kurt)


                # counts
                count_by_id += np.bincount(cc, minlength=max_id + 1)

                # % above threshold
                above = vv > high_threshold
                if np.any(above):
                    above_by_id += np.bincount(cc[above], minlength=max_id + 1)

                # p90 histogram per country
                for this_id in np.unique(cc):
                    vals = vv[cc == this_id]
                    hwin, _ = np.histogram(vals, bins=edges)
                    hist_by_id[this_id] += hwin.astype(np.int64)

    # build table with statistics
    out = lookup.copy()
    out["pixel_count"] = out["country_id"].map(lambda i: int(count_by_id[int(i)]))
    out["skewness_vulnerability"] = out["country_id"].map(_skewness)
    out["excess_kurtosis_vulnerability"] = out["country_id"].map(_kurtosis)


    # mean
    def compute_mean_vulnerability(country_id):
        i = int(country_id)
        if count_by_id[i] > 0:
            return float(sum_by_id[i] / count_by_id[i])
        else:
            return np.nan

    out["mean_vulnerability"] = out["country_id"].map(compute_mean_vulnerability)

    # variance = E[v^2] - (E[v])^2  
    def _variance(i):
        i = int(i)
        n = count_by_id[i]
        if n <= 0:
            return np.nan
        mean = sum_by_id[i] / n
        ev2 = sumsq_by_id[i] / n
        var = ev2 - mean * mean
        return float(max(var, 0.0))  # against tiny negative due to rounding

    out["variance_vulnerability"] = out["country_id"].map(_variance)
    out["std_vulnerability"] = np.sqrt(out["variance_vulnerability"])


    # p90
    def compute_percentile(country_id):
        i = int(country_id)
        return percentile_from_country_hist(hist_by_id[i], edges, pctl)

    out[f"p{int(pctl*100)}_vulnerability"] = out["country_id"].map(compute_percentile)


    # share above threshold
    def compute_share_above_threshold(country_id):
        i = int(country_id)
        if count_by_id[i] > 0:
            return float(above_by_id[i] / count_by_id[i])
        else:
            return np.nan

    out["share_area_above_threshold"] = out["country_id"].map(compute_share_above_threshold)


    # metadata
    out["high_threshold_used"] = high_threshold
    out["percentile_used"] = pctl

    out.to_csv(out_csv, index=False)
    print(f"{out_csv}")


In [ ]:
#RUN FUNCTIONS

# 1) Build country ID raster aligned to vulnerability raster grid and country lookup csv
build_country_id_raster(
    ref_path=VULN,
    gpkg_path=WORLD_COUNTRIES_GENERAL,
    out_raster=COUNTRY_ID,
    out_lookup_csv=LOOK_UP
)

# 2) Get global min and max inside countries
vmin, vmax = compute_global_minmax(VULN, COUNTRY_ID)

# 3) Choose "high vulnerability" threshold
if threshold_mode == "global_quantile":
    high_thr = global_threshold_from_hist(VULN, COUNTRY_ID, vmin, vmax, global_quantile)
    print(f"Global quantile threshold={global_quantile}: {high_thr}")
else:
    raise ValueError("threshold_mode must be 'global_quantile")

# 4) Compute statistics per country
compute_country_stats(
    VULN=VULN,
    country_id_path=COUNTRY_ID,
    lookup_csv=LOOK_UP,
    out_csv=STATS,
    vmin=vmin,
    vmax=vmax,
    pctl=pctl,
    high_threshold=high_thr,
)


In [ ]:
#Function to calculate different contributions to mean vulnerability scores per country
def compute_country_variable_contrib(country_id_path, lookup_csv, vuln_mean_path, out_csv):
    #read lookup csv
    lookup = pd.read_csv(lookup_csv)
    #find count of countries by reading max country id
    max_id = int(lookup["country_id"].max())

    # per country accumulators
    count = np.zeros(max_id + 1, dtype=np.int64)
    sum_v = np.zeros(max_id + 1, dtype=np.float64)

    sum_contrib = {f"sens_{k}": np.zeros(max_id + 1, dtype=np.float64) for k in sens_vars}
    sum_contrib.update({f"lack_{k}": np.zeros(max_id + 1, dtype=np.float64) for k in lackof_adapt_vars})

    # open all rasters 
    sens_srcs = {k: rasterio.open(p) for k, p in sens_vars.items()}
    lack_srcs = {k: rasterio.open(p) for k, p in lackof_adapt_vars.items()}
    vsrc = rasterio.open(vuln_mean_path)
    csrc = rasterio.open(country_id_path)

    try:
        #calculate number of windows
        n_rows = math.ceil(vsrc.height / window_size)
        n_cols = math.ceil(vsrc.width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, vsrc.width - x_off)
                h = min(window_size, vsrc.height - y_off)
                window = Window(x_off, y_off, w, h)

                cid = csrc.read(1, window=window)
                v = vsrc.read(1, window=window)

                #define valid values
                valid_v = (cid > 0) & np.isfinite(v) & (v != dst_nodata)#valid if inside countries, not no data, and finite number
                if not np.any(valid_v):
                    continue

                cc = cid[valid_v].astype(np.int32)
                vv = v[valid_v].astype(np.float32)

                count += np.bincount(cc, minlength=max_id + 1)
                sum_v += np.bincount(cc, weights=vv.astype(np.float64), minlength=max_id + 1)

                # sensitivity vars in this window
                sens_stack = []
                sens_valid_stack = []
                for k, src in sens_srcs.items():
                    a = src.read(1, window=window)
                    a_valid = np.isfinite(a) & (a != dst_nodata)
                    sens_stack.append(a)
                    sens_valid_stack.append(a_valid)

                # compute k_S only on valid_v pixels
                kS = np.zeros_like(v, dtype=np.float32)
                for a_valid in sens_valid_stack:
                    kS[valid_v] += a_valid[valid_v].astype(np.float32)
                # avoid division by zero (shouldn't happen if dimension always exists, but safe)
                kS[valid_v] = np.maximum(kS[valid_v], 1.0)

                # accumulate contributions for each sens var: 0.5 * a / kS
                for (k, a, a_valid) in zip(sens_vars.keys(), sens_stack, sens_valid_stack):
                    contrib = np.zeros_like(v, dtype=np.float32)
                    use = valid_v & a_valid
                    if np.any(use):
                        contrib[use] = 0.5 * (a[use].astype(np.float32) / kS[use])
                        sum_contrib[f"sens_{k}"] += np.bincount(
                            cid[use].astype(np.int32),
                            weights=contrib[use].astype(np.float64),
                            minlength=max_id + 1
                        )

                #lack of adaptive capacity vars in this window
                lack_stack = []
                lack_valid_stack = []
                for k, src in lack_srcs.items():
                    a = src.read(1, window=window)
                    a_valid = np.isfinite(a) & (a != dst_nodata)
                    lack_stack.append(a)
                    lack_valid_stack.append(a_valid)

                kL = np.zeros_like(v, dtype=np.float32)
                for a_valid in lack_valid_stack:
                    kL[valid_v] += a_valid[valid_v].astype(np.float32)
                kL[valid_v] = np.maximum(kL[valid_v], 1.0)

                for (k, a, a_valid) in zip(lackof_adapt_vars.keys(), lack_stack, lack_valid_stack):
                    contrib = np.zeros_like(v, dtype=np.float32)
                    use = valid_v & a_valid
                    if np.any(use):
                        contrib[use] = 0.5 * (a[use].astype(np.float32) / kL[use])
                        sum_contrib[f"lack_{k}"] += np.bincount(
                            cid[use].astype(np.int32),
                            weights=contrib[use].astype(np.float64),
                            minlength=max_id + 1
                        )

        out = lookup.copy()
        out["pixel_count"] = out["country_id"].map(lambda i: int(count[int(i)]))
        out["mean_vulnerability"] = out["country_id"].map(
            lambda i: float(sum_v[int(i)]/count[int(i)]) if count[int(i)]>0 else np.nan
        )

        # convert sums to means
        for key in sum_contrib:
            out[f"contrib_{key}"] = out["country_id"].map(
                lambda i: float(sum_contrib[key][int(i)]/count[int(i)]) if count[int(i)]>0 else np.nan
            )

        # totals
        sens_cols = [f"contrib_sens_{k}" for k in sens_vars]
        lack_cols = [f"contrib_lack_{k}" for k in lackof_adapt_vars]
        out["contrib_sensitivity_total"] = out[sens_cols].sum(axis=1)
        out["contrib_lack_total"] = out[lack_cols].sum(axis=1)
        out["contrib_sum_vars"] = out["contrib_sensitivity_total"] + out["contrib_lack_total"]
        out["diff_vs_vuln"] = out["contrib_sum_vars"] - out["mean_vulnerability"]

        out.to_csv(out_csv, index=False)
        print(out_csv)

    finally:
        for src in sens_srcs.values():
            src.close()
        for src in lack_srcs.values():
            src.close()
        vsrc.close()
        csrc.close()


In [ ]:
#run function to calculate contributions
compute_country_variable_contrib(
    country_id_path=COUNTRY_ID,
    lookup_csv=LOOK_UP,
    vuln_mean_path=VULN,
    out_csv=CONTRIBUTION,
)

In [ ]:
#plot 10 countries with highest mean vulnerability and different variables' contribution

# output path
out_png = "countries10_mean.png"

#input, read country variable contribution
df = pd.read_csv("country_variable_contrib.csv")

#choose 10 highest means
top10 = df.sort_values("mean_vulnerability", ascending=False).head(10).copy()

#define variables names using dictionaries
lackof_adapt_vars = list(lackof_adapt_vars.keys())

#define the column names in the CSV for contributions
sens_cols = [f"contrib_sens_{k}" for k in sens_vars]
lack_cols = [f"contrib_lack_{k}" for k in lackof_adapt_vars]

#create x positions for bars 
x = np.arange(len(top10))

#define labels for x axis
labels_x = top10["name"]

#define color ramps; two different ones (sensitivity and lack of adaptive capacity)
sens_cmap = plt.cm.inferno
lack_cmap = plt.cm.viridis
sens_colors = [sens_cmap(v) for v in np.linspace(0.15, 0.85, len(sens_cols))]
lack_colors = [lack_cmap(v) for v in np.linspace(0.15, 0.85, len(lack_cols))]

#create figure and axes
fig, ax = plt.subplots(figsize=(16, 8), dpi=200)
#set background to white
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

#tracks how tall the stacked bars already are
bottom = np.zeros(len(top10), dtype=np.float64)

# draw stacked pillars
#add all sensitivity contributions first (stacked)
for col, k, color in zip(sens_cols, sens_vars, sens_colors):
    #get contributions for this variable (one value per country)
    vals = top10[col].to_numpy(dtype=np.float64)
    #draw one stack layer for this variable
    plt.bar(x, vals, bottom=bottom, color=color, edgecolor="white", linewidth=0.3)
     #increase the "bottom" by this layer height
    bottom += np.nan_to_num(vals, nan=0.0)
# add all lack-of-adaptive-capacity contributions (stacked on top)
for col, k, color in zip(lack_cols, lackof_adapt_vars, lack_colors):
    vals = top10[col].to_numpy(dtype=np.float64)
    plt.bar(x, vals, bottom=bottom, color=color, edgecolor="white", linewidth=0.3)
    bottom += np.nan_to_num(vals, nan=0.0)

# Build legend 
handles, labels = [], []
#add a bold "Sensitivity" header line in the legend
handles.append(Line2D([], [], linestyle="none"))
labels.append(r"$\bf{Sensitivity}$")
#add one legend entry per sensitivity variable
for k, color in zip(sens_vars, sens_colors):
    handles.append(Line2D([0], [0], color=color, lw=6))
    labels.append(k)
#add an empty line spacer between groups
handles.append(Line2D([], [], linestyle="none"))
labels.append("")
#add a bold "Lack of adaptive capacity" header line
handles.append(Line2D([], [], linestyle="none"))
labels.append(r"$\bf{Lack\ of\ adaptive\ capacity}$")
#add one legend entry per lack of adaptive capacity variable
for k, color in zip(lackof_adapt_vars, lack_colors):
    handles.append(Line2D([0], [0], color=color, lw=6))
    labels.append(k)

#style axes
ax = plt.gca()
#remove top/right borders for cleaner look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
#soft grey axis lines
ax.spines["left"].set_color("#E3E6EA")
ax.spines["bottom"].set_color("#E3E6EA")

#y-axis label
plt.ylabel("Contribution to mean vulnerability",
    fontsize=16,
    color="#2B2F36",
)
#add title
plt.title("Top 10 countries with highest mean vulnerability",
    fontsize=18,
    fontweight="semibold",
    pad=14,
)

#x-axis tick labels 
plt.xticks(
    x,
    labels_x,
    rotation=60,
    ha="right",
    fontsize=16,
    color="#2B2F36",
)

# #set y-axis limit based on total stack height
plt.ylim(0, float(np.nanmax(bottom)) * 1.05)

#small x padding
plt.margins(x=0.01)

#add legend  
plt.legend(handles,
    labels,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=12,
    labelcolor="#2B2F36",
)

#save plot as png and show plot
plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# plot country classes of mean vulnerability (EPSG:6933)

# paths
stats_csv = STATS
countries_path = WORLD_COUNTRIES_GENERAL
out_png = "country_mean_vulnerability_classes.png"

target_crs = "EPSG:6933"

# load data
stats = pd.read_csv(stats_csv).copy()

world = gpd.read_file(countries_path)
world = world.to_crs(target_crs)

# drop Antarctica
world = world[world["COUNTRY"] != "Antarctica"].copy()

# standardize join key
stats["iso"] = stats["iso"].astype(str).str.strip().str.upper()
world["iso"] = world["ISO"].astype(str).str.strip().str.upper()

# merge stats
plot_gdf = world.merge(
    stats[["iso", "name", "mean_vulnerability"]],
    on="iso",
    how="left"
)

# classify vulnerability (quantiles)
class_names = ["Very low", "Low", "Moderate", "High", "Very high"]

qcut_result = pd.qcut(
    plot_gdf["mean_vulnerability"],
    q=5,
    labels=class_names,
    duplicates="drop"
)

plot_gdf["vuln_class"] = qcut_result

# get numeric class breaks
intervals = pd.qcut(
    plot_gdf["mean_vulnerability"].dropna(),
    q=5,
    duplicates="drop"
).cat.categories

breaks = [iv.left for iv in intervals] + [intervals[-1].right]

# force first lower bound to 0
breaks[0] = 0.0

# round to 2 decimals for legend
breaks = [round(b, 2) for b in breaks]

# build non-overlapping legend labels
legend_labels = [
    f"Very low ≤ {breaks[1]:.2f}",
    f"Low < {breaks[1]:.2f} to ≤ {breaks[2]:.2f}",
    f"Moderate < {breaks[2]:.2f} to ≤ {breaks[3]:.2f}",
    f"High < {breaks[3]:.2f} to ≤ {breaks[4]:.2f}",
    f"Very high < {breaks[4]:.2f}",
]

# color palette
class_colors = {
    "Very low": "#FFF1F2",
    "Low": "#FECACA",
    "Moderate": "#F87171",
    "High": "#DC2626",
    "Very high": "#7F1D1D",
}

nodata_color = "#D1D5DB"   # grey for missing data

# create figure
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# base countries
world.plot(
    ax=ax,
    facecolor=nodata_color,
    edgecolor="#FFFFFF",
    linewidth=0.35,
    zorder=1
)

# plot classes
for cls in class_names:
    subset = plot_gdf[plot_gdf["vuln_class"] == cls]
    if len(subset) > 0:
        subset.plot(
            ax=ax,
            facecolor=class_colors[cls],
            edgecolor="#FFFFFF",
            linewidth=0.35,
            zorder=2
        )
        
# plot missing countries (no data)
missing = plot_gdf[plot_gdf["mean_vulnerability"].isna()]
if len(missing) > 0:
    missing.plot(
        ax=ax,
        facecolor=nodata_color,
        edgecolor="#FFFFFF",
        linewidth=0.35,
        zorder=2
    )

# country borders
world.boundary.plot(
    ax=ax,
    color="#695C5A",
    linewidth=0.3,
    zorder=3
)

# title
ax.set_title(
    "Socio-Ecological Vulnerability to Mining",
    fontsize=18,
    fontweight="semibold",
    pad=14
)

ax.set_axis_off()

# legend
handles = [
    Patch(facecolor=class_colors["Very low"], edgecolor="none", label=legend_labels[0]),
    Patch(facecolor=class_colors["Low"], edgecolor="none", label=legend_labels[1]),
    Patch(facecolor=class_colors["Moderate"], edgecolor="none", label=legend_labels[2]),
    Patch(facecolor=class_colors["High"], edgecolor="none", label=legend_labels[3]),
    Patch(facecolor=class_colors["Very high"], edgecolor="none", label=legend_labels[4]),
]

if plot_gdf["mean_vulnerability"].isna().any():
    handles.append(Patch(facecolor=nodata_color, edgecolor="none", label="No data"))

leg = ax.legend(
    handles=handles,
    title="Mean Vulnerability",
    loc="lower left",
    frameon=True,
    facecolor="white",
    edgecolor="#E3E6EA",
    fontsize=10,
    title_fontsize=11
)

# match text color styling
for txt in leg.get_texts():
    txt.set_color("#2B2F36")
leg.get_title().set_color("#2B2F36")

# save
plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()